In [18]:
from pathlib import Path
import pandas as pd

source_name = "netflix_combined_data_new.csv"
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data" / "raw" / source_name).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"No se encontró data/raw/{source_name}. Ejecuta el notebook dentro del proyecto."
    )

source_path = project_root / "data" / "raw" / source_name
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
df_original = pd.read_csv(source_path, encoding="utf-8-sig", low_memory=False)

print(f"Fuente: {source_path.relative_to(project_root)}")
print(f"Filas: {len(df_original):,} | Columnas: {len(df_original.columns)}")
display(df_original.head())

Fuente: data\raw\netflix_combined_data_new.csv
Filas: 32,000 | Columnas: 18


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0


## 1. Diagnóstico de entrada

Antes de transformar, revisamos tipos, faltantes y posibles duplicados. Los faltantes estructurales se conservan: por ejemplo, presupuesto e ingresos no aplican a series y duración no aplica a películas en esta fuente.

In [23]:
print("Dimensiones:", df_original.shape)
print("Tipos de datos:")
display(df_original.dtypes.rename("tipo").to_frame())

missing_before = pd.DataFrame(
    {
        "faltantes": df_original.isna().sum(),
        "porcentaje": (df_original.isna().mean() * 100).round(2),
    }
).sort_values("faltantes", ascending=False)
print("Valores faltantes:")
display(missing_before)
print(f"Filas exactamente duplicadas: {df_original.duplicated().sum():,}")
print(f"IDs de origen repetidos: {df_original['show_id'].duplicated().sum():,}")

Dimensiones: (32000, 18)
Tipos de datos:


,tipo
show_id,int64
type,str
title,str
director,str
cast,str
country,str
date_added,str
release_year,int64
rating,float64
duration,str


Valores faltantes:


,faltantes,porcentaje
budget,16000,50.00
duration,16000,50.00
revenue,16000,50.00
director,11097,34.68
description,3338,10.43
country,2263,7.07
cast,1361,4.25
genres,1081,3.38
type,0,0.00
show_id,0,0.00


Filas exactamente duplicadas: 0
IDs de origen repetidos: 406


## 2. Limpieza y preparación

Se normalizan nombres y espacios, se corrigen secuencias de texto con codificación dañada solo si la conversión es segura, y se convierten columnas a sus tipos analíticos. Se elimina una puntuación duplicada únicamente si `rating` y `vote_average` siguen siendo idénticas en todos los registros.

In [20]:
df = df_original.copy()
input_rows = len(df)

df.columns = (
    df.columns.astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
required_columns = {"show_id", "type", "title", "date_added", "release_year"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(missing_columns)}")

def repair_mojibake(value: str) -> str:
    markers = ("Ã", "Â", "â€", "ï¿½")
    if not any(marker in value for marker in markers):
        return value
    try:
        repaired = value.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value
    if sum(repaired.count(marker) for marker in markers) < sum(
        value.count(marker) for marker in markers
    ):
        return repaired
    return value

text_columns = df.select_dtypes(include=["object", "string"]).columns
repaired_text_values = 0
for column in text_columns:
    original_values = df[column].astype("string")
    repaired_values = original_values.map(
        lambda value: repair_mojibake(value) if pd.notna(value) else value
    )
    repaired_text_values += int((original_values != repaired_values).sum())
    df[column] = repaired_values.str.strip().replace("", pd.NA)

type_normalized = df["type"].str.casefold()
df["type"] = type_normalized.map({"movie": "Movie", "tv show": "TV Show"}).fillna(df["type"])

rows_before_deduplication = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before_deduplication - len(df)

numeric_columns = [
    "show_id", "release_year", "rating", "popularity", "vote_count",
    "vote_average", "budget", "revenue",
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

if {"rating", "vote_average"}.issubset(df.columns):
    if not df["rating"].equals(df["vote_average"]):
        raise ValueError("rating y vote_average ya no coinciden; no se eliminará ninguna columna.")
    df = df.drop(columns="rating")

for column in ["show_id", "release_year", "vote_count"]:
    if column in df.columns:
        df[column] = df[column].astype("Int64")

date_values_before = df["date_added"].notna()
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
invalid_dates = int((date_values_before & df["date_added"].isna()).sum())

if "duration" in df.columns:
    duration_parts = df["duration"].astype("string").str.extract(r"^\s*(\d+)\s*(.*?)\s*$")
    df["duration_value"] = pd.to_numeric(duration_parts[0], errors="coerce").astype("Int64")
    df["duration_unit"] = duration_parts[1].str.strip().str.casefold().replace("", pd.NA)

id_occurrence = df.groupby(["type", "show_id"], dropna=False).cumcount().add(1).astype("string")
type_slug = df["type"].astype("string").str.casefold().str.replace(r"\s+", "_", regex=True)
id_text = df["show_id"].astype("string").fillna("sin_id")
df["content_key"] = type_slug + "_" + id_text + "_" + id_occurrence
df = df.reset_index(drop=True)

print(f"Filas de entrada: {input_rows:,}")
print(f"Duplicados exactos eliminados: {duplicates_removed:,}")
print(f"Valores de texto reparados: {repaired_text_values:,}")
print(f"Fechas inválidas convertidas a nulo: {invalid_dates:,}")
print(f"Filas listas para validar: {len(df):,}")

Filas de entrada: 32,000
Duplicados exactos eliminados: 0
Valores de texto reparados: 1
Fechas inválidas convertidas a nulo: 0
Filas listas para validar: 32,000


## 3. Validaciones de calidad

Se verifica que la llave de contenido sea única, que no falten título ni tipo, que las puntuaciones estén entre 0 y 10 y que las métricas no sean negativas. Los nulos restantes se reportan, no se rellenan artificialmente.

In [21]:
assert df["title"].notna().all(), "Hay registros sin título."
assert df["type"].notna().all(), "Hay registros sin tipo."
assert df["content_key"].is_unique, "content_key debe ser único por registro."
assert "rating" not in df.columns, "Se esperaba conservar solo vote_average."

invalid_scores = int(
    (df["vote_average"].notna() & ~df["vote_average"].between(0, 10)).sum()
)
negative_metrics = {
    column: int((df[column].notna() & df[column].lt(0)).sum())
    for column in ["popularity", "vote_count", "budget", "revenue"]
    if column in df.columns
}
assert invalid_scores == 0, "Hay puntuaciones fuera del rango 0-10."
assert sum(negative_metrics.values()) == 0, "Hay métricas negativas que requieren revisión."

quality_summary = pd.DataFrame(
    [
        ("Filas de entrada", input_rows),
        ("Filas de salida", len(df)),
        ("Duplicados exactos eliminados", duplicates_removed),
        ("Claves duplicadas", int(df["content_key"].duplicated().sum())),
        ("Fechas inválidas", invalid_dates),
        ("Valores de texto reparados", repaired_text_values),
        ("Puntuaciones fuera de 0-10", invalid_scores),
        ("Métricas negativas", sum(negative_metrics.values())),
    ],
    columns=["Control", "Resultado"],
)
missing_report = pd.DataFrame(
    {
        "columna": df.columns,
        "faltantes": [int(df[column].isna().sum()) for column in df.columns],
        "porcentaje_faltante": [round(float(df[column].isna().mean() * 100), 2) for column in df.columns],
    }
).sort_values("porcentaje_faltante", ascending=False)

display(quality_summary)
display(missing_report)

,Control,Resultado
0,Filas de entrada,32000
1,Filas de salida,32000
2,Duplicados exactos eliminados,0
3,Claves duplicadas,0
4,Fechas inválidas,0
5,Valores de texto reparados,1
6,Puntuaciones fuera de 0-10,0
7,Métricas negativas,0


,columna,faltantes,porcentaje_faltante
15,budget,16000,50.00
18,duration_unit,16000,50.00
17,duration_value,16000,50.00
8,duration,16000,50.00
16,revenue,16000,50.00
3,director,11097,34.68
11,description,3340,10.44
5,country,2263,7.07
4,cast,1361,4.25
9,genres,1081,3.38


## 4. Exportación para Power BI o Looker Studio

El notebook genera un único archivo de datos tratado. El original permanece intacto y el resumen de calidad se muestra aquí, sin crear CSV auxiliares.

In [22]:
output_path = processed_dir / "netflix_catalog_clean.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")

print(f"Dataset limpio: {output_path.relative_to(project_root)}")
print(f"Dimensiones: {len(df):,} filas x {len(df.columns)} columnas")
print(f"Llave única: {df['content_key'].is_unique}")
print(f"rating eliminada; vote_average conservada: {'rating' not in df.columns and 'vote_average' in df.columns}")
display(df.head())

Dataset limpio: data\processed\netflix_catalog_clean.csv
Dimensiones: 32,000 filas x 20 columnas
Llave única: True
rating eliminada; vote_average conservada: True


,show_id,type,title,director,cast,country,date_added,release_year,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue,duration_value,duration_unit,content_key
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,<NA>,<NA>,movie_10192_1
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,<NA>,<NA>,movie_27205_1
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0,<NA>,<NA>,movie_12444_1
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0,<NA>,<NA>,movie_38757_1
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0,<NA>,<NA>,movie_10191_1
